# 01_proceso_pln1_anonimizacion_normalizacion con trazabilidad ECG

**Objetivo:** construir la cohorte clínica anonimizada y procesada, preservando una trazabilidad privada que permita integrar posteriormente los parámetros ECG extraídos desde PDF.

Flujo corregido:

```text
Base de Datos Original.xlsx
        ↓
01 anonimización + normalización + PLN/NLP + flags clínicos
        ↓
Base_Datos_Original_Anonimizada_Procesada_TFM.xlsx
        ↓
Crosswalk privado para integración ECG
        ↓
crosswalk_paciente_ecg.xlsx
```

La salida analítica no conserva identificadores directos. La tabla `crosswalk_paciente_ecg.xlsx` es un artefacto privado de trazabilidad.


In [1]:
# Instalación de dependencias requeridas
# Ejecutar esta celda si el entorno no tiene las dependencias instaladas.

import importlib.util
import subprocess
import sys

DEPENDENCIAS = {
    "openpyxl": "openpyxl",
    "xlsxwriter": "XlsxWriter",
}

for modulo, paquete in DEPENDENCIAS.items():
    if importlib.util.find_spec(modulo) is None:
        print(f"Instalando dependencia faltante: {paquete}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", paquete])
    else:
        print(f"Dependencia disponible: {modulo}")


Dependencia disponible: openpyxl
Dependencia disponible: xlsxwriter


In [2]:
from pathlib import Path
import hashlib
import os
import re
import unicodedata
from datetime import datetime

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 200)


## 1. Configuración de rutas

El archivo original debe mantenerse fuera del repositorio público. Las rutas se definen de forma relativa para facilitar ejecución local desde la carpeta de notebooks.


In [3]:
# Entradas
INPUT_PATH = Path("Base de Datos Original.xlsx")

# Salidas analíticas
OUTPUT_XLSX = Path("Base_Datos_Original_Anonimizada_Procesada_TFM.xlsx")
DICT_CSV = Path("Diccionario_Normalizacion_Antecedentes.csv")
LOG_TXT = Path("Log_Transformacion_Cohorte_TFM.txt")

# Salidas privadas de trazabilidad. NO PUBLICAR.
CROSSWALK_XLSX = Path("crosswalk_paciente_ecg.xlsx")
AUDITORIA_PRIVADA_XLSX = Path("Auditoria_Privada_Trazabilidad_TFM.xlsx")

# Referencia opcional para validación, si existe localmente.
REFERENCE_XLSX = Path("Base_Datos_Original_Anonimizada_Procesada_TFM_REFERENCIA.xlsx")

print("Entrada:", INPUT_PATH.resolve())
print("Salida analítica:", OUTPUT_XLSX.resolve())
print("Crosswalk privado:", CROSSWALK_XLSX.resolve())


Entrada: E:\Capacitacion 2025\Magister en Inteligencia Artificial\99 Trabajo Fin de Estudios TFE\GitHub\tfm-ecg-riesgo-cardiometabolico\notebooks\Base de Datos Original.xlsx
Salida analítica: E:\Capacitacion 2025\Magister en Inteligencia Artificial\99 Trabajo Fin de Estudios TFE\GitHub\tfm-ecg-riesgo-cardiometabolico\notebooks\Base_Datos_Original_Anonimizada_Procesada_TFM.xlsx
Crosswalk privado: E:\Capacitacion 2025\Magister en Inteligencia Artificial\99 Trabajo Fin de Estudios TFE\GitHub\tfm-ecg-riesgo-cardiometabolico\notebooks\crosswalk_paciente_ecg.xlsx


## 2. Funciones auxiliares de normalización

Estas funciones se usan tanto para la anonimización clínica como para construir una clave de integración compatible con el dataset ECG generado desde PDF.


In [4]:
def strip_accents(value):
    """Elimina tildes y diacríticos. Devuelve cadena vacía para valores nulos."""
    if pd.isna(value):
        return ""
    return "".join(
        char for char in unicodedata.normalize("NFD", str(value))
        if unicodedata.category(char) != "Mn"
    )


def clean_text(value):
    """Normaliza texto clínico libre para búsqueda por patrones."""
    value = strip_accents(value).lower()
    value = re.sub(r"[\n\r\t]", " ", value)
    value = re.sub(r"[/,+;|(){}\[\]:=._-]", " ", value)
    value = re.sub(r"\s+", " ", value).strip()
    return value


def normalize_name(value):
    """Normaliza nombres para matching con archivos ECG: mayúsculas, sin acentos y espacios colapsados."""
    if pd.isna(value):
        return pd.NA
    text = strip_accents(str(value))
    text = re.sub(r"\.pdf$", "", text, flags=re.IGNORECASE)
    text = re.sub(r"[\s_\-]+(\d+)\s*$", "", text)
    text = re.sub(r"[^A-Za-zÑñ\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip().upper()
    return text if text else pd.NA


def normalize_rut(value):
    """Normaliza RUT para uso interno privado. No debe conservarse en la base analítica."""
    if pd.isna(value):
        return pd.NA
    text = str(value).upper().strip()
    text = re.sub(r"[^0-9K]", "", text)
    return text if text else pd.NA


def parse_fecha_segura(value):
    """Parsea fechas evitando ambigüedad entre formatos ISO y formatos día/mes/año."""
    if pd.isna(value):
        return pd.NaT

    if isinstance(value, (pd.Timestamp, datetime)):
        return pd.to_datetime(value, errors="coerce")

    s = str(value).strip()
    if not s:
        return pd.NaT

    # Formato ISO: YYYY-MM-DD o YYYY-MM-DD HH:MM:SS[.ffffff]
    if re.match(r"^\d{4}-\d{2}-\d{2}", s):
        return pd.to_datetime(s, errors="coerce", dayfirst=False)

    # Formato latino: DD-MM-YYYY, DD/MM/YYYY u otros formatos con día primero.
    return pd.to_datetime(s, errors="coerce", dayfirst=True)


def normalize_date(value):
    """Convierte fechas a formato ISO YYYY-MM-DD."""
    date = parse_fecha_segura(value)
    if pd.isna(date):
        return pd.NA
    return date.strftime("%Y-%m-%d")


def extract_number(value):
    """Extrae el primer número encontrado en una cadena clínica."""
    if pd.isna(value):
        return np.nan
    match = re.search(r"[-+]?\d+([.,]\d+)?", strip_accents(str(value)))
    return float(match.group(0).replace(",", ".")) if match else np.nan


def normalize_yes_no(value):
    """Convierte variantes textuales de sí/no a 1/0."""
    text = clean_text(value)
    if text in ["si", "sí", "s", "yes", "y", "positivo", "presente"] or re.search(r"\bsi\b", text):
        return 1
    if text in ["no", "n", "negativo", "ausente"] or re.search(r"\bno\b", text):
        return 0
    return np.nan


def sha256_private(value, salt=None):
    """Genera hash privado reproducible. El salt no debe publicarse."""
    if pd.isna(value) or str(value).strip() == "":
        return pd.NA
    salt = salt or os.environ.get("TFM_PRIVATE_SALT", "CAMBIAR_SALT_PRIVADO_LOCAL")
    payload = f"{str(value).strip()}|{salt}".encode("utf-8")
    return hashlib.sha256(payload).hexdigest()


## 3. Detección de columnas críticas

El notebook intenta detectar columnas de nombre, RUT, fecha de nacimiento y fecha de atención usando nombres frecuentes. Si la base original usa nombres distintos, se deben ajustar las listas de candidatos.


In [5]:
NAME_CANDIDATES = [
    "Nombre", "Paciente", "NombrePaciente", "Nombre Paciente", "Nombres", "NombreCompleto", "Nombre Completo"
]

RUT_CANDIDATES = [
    "Rut", "RUT", "RUN", "Documento", "Identificador"
]

BIRTHDATE_CANDIDATES = [
    "FechaNacimiento", "Fecha Nacimiento", "FecNacimiento", "Nacimiento"
]

ATTENTION_DATE_CANDIDATES = [
    "UltimaFechaAtencion", "FechaAtencion", "Fecha Atención", "Fecha", "FechaExamen", "Fecha Examen", "FechaOrden"
]


def first_existing_column(df, candidates):
    """Devuelve la primera columna existente entre una lista de candidatos."""
    normalized = {str(column).strip().lower(): column for column in df.columns}
    for candidate in candidates:
        key = candidate.strip().lower()
        if key in normalized:
            return normalized[key]
    return None


## 4. Reglas PLN/NLP para antecedentes médicos

Las categorías clínicas derivadas se mantienen respecto del pipeline original.


In [6]:
CATEGORIES = {
    "ANT_HTA": [r"hta", r"hipertension", r"hipertenso", r"hipertensa", r"presion alta"],
    "ANT_DIABETES": [r"dm", r"dm2", r"diabetes", r"diabetico", r"diabetica"],
    "ANT_DISLIPIDEMIA": [r"dlp", r"dislipidemia", r"hiperlipidemia", r"colesterol", r"hipercolesterolemia"],
    "ANT_TABAQUISMO": [r"tabaquismo", r"fumador", r"fumadora", r"tabaco", r"cigarr"],
    "ANT_OBESIDAD": [r"obesidad", r"obeso", r"obesa"],
    "ANT_INFARTO": [r"iam", r"infarto", r"miocardio"],
    "ANT_ACV": [r"acv", r"accidente cerebrovascular", r"ictus", r"ave"],
    "ANT_CARDIOPATIA": [r"cardiopatia", r"coronaria", r"cardiaca", r"cardiaco", r"angina"],
    "ANT_ARRITMIA": [r"arritmia", r"fibrilacion", r"fa"],
    "ANT_ENF_RENAL": [r"irc", r"renal", r"rinon", r"riñon", r"nefropatia"],
    "ANT_EPOC": [r"epoc", r"enfermedad pulmonar obstructiva"],
    "ANT_CANCER": [r"cancer", r"tumor", r"neoplasia"],
    "ANT_HIPOTIROIDISMO": [r"hipotiroidismo", r"hipotiroid"],
    "ANT_SALUD_MENTAL": [r"depresion", r"ansiedad", r"psiquiatr", r"bipolar", r"trastorno"],
    "ANT_ALCOHOL": [r"alcohol", r"bebedor", r"bebedora"],
    "ANT_ASMA": [r"asma", r"asmatico", r"asmatica"],
}

NEGATION_TERMS = [
    "no", "niega", "sin", "descarta", "no refiere",
    "sin antecedentes", "no presenta", "no registra", "negativo"
]


def detect_category(text, patterns):
    """Detecta una categoría clínica en texto libre evitando falsos positivos por negación local."""
    text = clean_text(text)
    if not text:
        return 0

    for pattern in patterns:
        for match in re.finditer(pattern, text):
            context = text[max(0, match.start() - 35):match.start()]
            negated = any(re.search(rf"{re.escape(term)}", context) for term in NEGATION_TERMS)
            if not negated:
                return 1
    return 0


## 5. Construcción de identificadores y crosswalk privado

`PACIENTE_ID` se asigna antes de eliminar identificadores. La base analítica conserva `PACIENTE_ID`, pero no conserva nombres, RUT, fecha de nacimiento ni antecedentes médicos en texto libre.

La tabla `crosswalk_paciente_ecg.xlsx` conserva la relación privada necesaria para integrar ECG posteriormente.


In [7]:
def build_private_traceability(df):
    """Construye PACIENTE_ID, REGISTRO_ID y crosswalk privado para integración ECG."""
    df = df.copy()
    df.columns = [str(column).strip() for column in df.columns]

    name_col = first_existing_column(df, NAME_CANDIDATES)
    rut_col = first_existing_column(df, RUT_CANDIDATES)
    birth_col = first_existing_column(df, BIRTHDATE_CANDIDATES)
    attention_date_col = first_existing_column(df, ATTENTION_DATE_CANDIDATES)

    trace = pd.DataFrame(index=df.index)
    trace["REGISTRO_ID"] = [f"REG_{index:06d}" for index in range(1, len(df) + 1)]

    if name_col:
        trace["nombre_paciente_norm"] = df[name_col].map(normalize_name)
    else:
        trace["nombre_paciente_norm"] = pd.NA

    if rut_col:
        trace["rut_norm"] = df[rut_col].map(normalize_rut)
    else:
        trace["rut_norm"] = pd.NA

    if birth_col:
        trace["fecha_nacimiento_norm"] = df[birth_col].map(normalize_date)
    else:
        trace["fecha_nacimiento_norm"] = pd.NA

    if attention_date_col:
        trace["fecha_atencion_norm"] = df[attention_date_col].map(normalize_date)
    else:
        trace["fecha_atencion_norm"] = pd.NA

    # Identidad privada para asignar PACIENTE_ID. Prioridad: RUT; luego nombre+fecha nacimiento; luego nombre; luego registro.
    person_key = pd.Series(pd.NA, index=df.index, dtype="object")

    if rut_col:
        person_key = trace["rut_norm"].where(trace["rut_norm"].notna(), person_key)

    name_birth_key = (
        trace["nombre_paciente_norm"].astype("string").fillna("") + "|" +
        trace["fecha_nacimiento_norm"].astype("string").fillna("")
    )
    valid_name_birth = trace["nombre_paciente_norm"].notna() & trace["fecha_nacimiento_norm"].notna()
    person_key = person_key.where(person_key.notna(), name_birth_key.where(valid_name_birth, pd.NA))

    person_key = person_key.where(person_key.notna(), trace["nombre_paciente_norm"])
    person_key = person_key.where(person_key.notna(), trace["REGISTRO_ID"])

    patient_map = {key: f"PAC_{idx:06d}" for idx, key in enumerate(pd.Series(person_key).drop_duplicates(), start=1)}
    trace["PACIENTE_ID"] = person_key.map(patient_map)

    trace["clave_matching"] = np.where(
        trace["nombre_paciente_norm"].notna() & trace["fecha_atencion_norm"].notna(),
        trace["nombre_paciente_norm"].astype(str) + " | " + trace["fecha_atencion_norm"].astype(str),
        pd.NA,
    )

    trace["clave_hash_privada"] = trace["clave_matching"].map(sha256_private)
    trace["columna_nombre_detectada"] = name_col or "NO_DETECTADA"
    trace["columna_rut_detectada"] = rut_col or "NO_DETECTADA"
    trace["columna_fecha_nacimiento_detectada"] = birth_col or "NO_DETECTADA"
    trace["columna_fecha_atencion_detectada"] = attention_date_col or "NO_DETECTADA"
    trace["estado_validacion"] = np.where(trace["clave_matching"].notna(), "pendiente_validacion", "sin_clave_matching")
    trace["metodo_match"] = np.where(trace["clave_matching"].notna(), "nombre_normalizado_fecha_atencion", "no_disponible")
    trace["observaciones"] = ""

    private_columns = [
        "REGISTRO_ID", "PACIENTE_ID", "nombre_paciente_norm", "fecha_atencion_norm", "clave_matching",
        "clave_hash_privada", "estado_validacion", "metodo_match", "observaciones",
        "rut_norm", "fecha_nacimiento_norm", "columna_nombre_detectada", "columna_rut_detectada",
        "columna_fecha_nacimiento_detectada", "columna_fecha_atencion_detectada",
    ]
    trace = trace[private_columns]

    detected = {
        "name_col": name_col,
        "rut_col": rut_col,
        "birth_col": birth_col,
        "attention_date_col": attention_date_col,
    }
    return trace, detected


## 6. Construcción de cohorte anonimizada y procesada

Esta función conserva la lógica previa de normalización clínica y agrega `PACIENTE_ID` y `REGISTRO_ID` generados en la trazabilidad privada.


In [8]:
def build_cohort(input_path=INPUT_PATH):
    """Construye cohorte anonimizada, procesada y crosswalk privado."""
    df = pd.read_excel(input_path, sheet_name=0)
    df.columns = [str(column).strip() for column in df.columns]

    trace, detected = build_private_traceability(df)

    out = pd.DataFrame({
        "REGISTRO_ID": trace["REGISTRO_ID"],
        "PACIENTE_ID": trace["PACIENTE_ID"],
    })

    direct_identifiers = [
        column for column in [detected["name_col"], detected["rut_col"], detected["birth_col"]]
        if column is not None and column in df.columns
    ]

    attention_col = detected["attention_date_col"]
    if attention_col:
        dates = df[attention_col].apply(parse_fecha_segura)
        out["UltimaAtencion_Anio"] = dates.dt.year
        out["UltimaAtencion_Mes"] = dates.dt.month
        out["DiasDesdePrimeraAtencion"] = (dates - dates.min()).dt.days

    exclude = set(direct_identifiers + ["AntecedentesMedicos"])
    if attention_col:
        exclude.add(attention_col)

    for column in df.columns:
        if column not in exclude:
            out[column] = df[column]

    numeric_columns = {
        "Peso": "Peso_kg",
        "Altura": "Altura_m",
        "IMC": "IMC_num",
        "PA_Sistolica": "PA_Sistolica_mmHg",
        "PA_Diastolica": "PA_Diastolica_mmHg",
        "Glicemia": "Glicemia_mg_dl",
        "ColesterolTotal": "ColesterolTotal_mg_dl",
        "HDL": "HDL_mg_dl",
        "LDL": "LDL_mg_dl",
        "Trigliceridos": "Trigliceridos_mg_dl",
        "Hemoglobina": "Hemoglobina_gr_pct",
        "Creatinina": "Creatinina_mg_dl",
    }

    for source, target in numeric_columns.items():
        if source in df.columns:
            out[target] = df[source].map(extract_number)

    if "Tabaquismo" in df.columns:
        out["Tabaquismo_bin"] = df["Tabaquismo"].map(normalize_yes_no)

    if "Diabetes" in df.columns:
        out["Diabetes_bin"] = df["Diabetes"].map(normalize_yes_no)

    raw_antecedents = (
        df["AntecedentesMedicos"].fillna("").astype(str)
        if "AntecedentesMedicos" in df.columns
        else pd.Series([""] * len(df), index=df.index)
    )

    normalized_labels = []
    category_arrays = {category: np.zeros(len(df), dtype=int) for category in CATEGORIES}

    for index, text in enumerate(raw_antecedents):
        detected_labels = []
        for category, patterns in CATEGORIES.items():
            value = detect_category(text, patterns)
            category_arrays[category][index] = value
            if value:
                detected_labels.append(category.replace("ANT_", "").lower())
        normalized_labels.append(", ".join(detected_labels))

    for category, values in category_arrays.items():
        out[category] = values

    out["AntecedentesMedicos_Normalizado"] = normalized_labels
    out["AntecedentesMedicos_TextoOriginal_Presente"] = raw_antecedents.map(
        lambda value: 1 if clean_text(value) else 0
    )

    def ge(column, threshold):
        return np.where(pd.to_numeric(out[column], errors="coerce") >= threshold, 1, 0)

    if "PA_Sistolica_mmHg" in out:
        out["FLAG_PA_SISTOLICA_ALTA"] = ge("PA_Sistolica_mmHg", 140)
    if "PA_Diastolica_mmHg" in out:
        out["FLAG_PA_DIASTOLICA_ALTA"] = ge("PA_Diastolica_mmHg", 90)
    if "Glicemia_mg_dl" in out:
        out["FLAG_GLICEMIA_ALTA"] = ge("Glicemia_mg_dl", 126)
    if "LDL_mg_dl" in out:
        out["FLAG_LDL_ALTO"] = ge("LDL_mg_dl", 130)
    if "Trigliceridos_mg_dl" in out:
        out["FLAG_TRIGLICERIDOS_ALTOS"] = ge("Trigliceridos_mg_dl", 150)
    if "IMC_num" in out:
        out["FLAG_OBESIDAD_IMC"] = ge("IMC_num", 30)

    if "HDL_mg_dl" in out and "Sexo" in out:
        female = out["Sexo"].astype(str).str.lower().str.contains("femen|female|mujer|f", na=False)
        out["FLAG_HDL_BAJO"] = np.where(
            (female & (pd.to_numeric(out["HDL_mg_dl"], errors="coerce") < 50)) |
            (~female & (pd.to_numeric(out["HDL_mg_dl"], errors="coerce") < 40)),
            1,
            0
        )

    return df, out, trace, direct_identifiers, detected


## 7. Exportación de salidas

Se generan salidas separadas:

- base clínica anonimizada y procesada;
- diccionario de variables derivadas;
- log técnico;
- crosswalk privado para integración ECG;
- auditoría privada de trazabilidad.


In [9]:
def export_outputs(df, out, trace, direct_identifiers, detected):
    """Exporta cohorte procesada, diccionario, logs y crosswalk privado."""
    clinical_binary_cols = [column for column in out.columns if column.startswith("ANT_")]
    flag_cols = [column for column in out.columns if column.startswith("FLAG_")]

    summary = pd.DataFrame([
        ["Filas originales", len(df)],
        ["Columnas originales", df.shape[1]],
        ["Filas finales", len(out)],
        ["Columnas finales", out.shape[1]],
        ["Pacientes únicos", out["PACIENTE_ID"].nunique()],
        ["Registros únicos", out["REGISTRO_ID"].nunique()],
        ["Identificadores directos eliminados", ", ".join(direct_identifiers)],
        ["Variables de antecedentes generadas", len(clinical_binary_cols)],
        ["Flags cardiometabólicos generados", len(flag_cols)],
        ["Texto original AntecedentesMedicos conservado", "No"],
        ["Crosswalk privado generado", str(CROSSWALK_XLSX)],
        ["Registros con clave_matching", int(trace["clave_matching"].notna().sum())],
    ], columns=["Metrica", "Valor"])

    anonymization = pd.DataFrame([
        ["Nombre/Paciente", "Eliminado de base analítica", "Identificador directo; conservado solo normalizado en crosswalk privado si existe"],
        ["Rut/RUN", "Eliminado de base analítica", "Identificador nacional directo; usado solo para asignación privada de PACIENTE_ID si existe"],
        ["FechaNacimiento", "Eliminado de base analítica", "Cuasi-identificador; usado solo en trazabilidad privada si existe"],
        ["FechaAtencion", "Generalizado", "Año, mes y días relativos en la base analítica; fecha ISO solo en crosswalk privado"],
        ["AntecedentesMedicos", "Derivado y no conservado", "Texto clínico libre sensible"],
        ["PACIENTE_ID", "Creado", "Identificador anónimo estable dentro del pipeline"],
        ["REGISTRO_ID", "Creado", "Identificador anónimo de fila o evento clínico"],
        ["clave_matching", "Solo crosswalk privado", "Clave nombre normalizado + fecha para integración ECG"],
    ], columns=["Campo original", "Tratamiento", "Justificacion"])

    dictionary = pd.DataFrame([{
        "Variable": category,
        "Descripcion": category.replace("ANT_", "").replace("_", " ").lower(),
        "Patrones": " | ".join(patterns),
        "Tipo": "Binaria derivada de AntecedentesMedicos",
    } for category, patterns in CATEGORIES.items()])

    counts = pd.DataFrame([{
        "Variable": column,
        "Positivos": int(pd.to_numeric(out[column], errors="coerce").fillna(0).sum()),
        "Pct": round(float(pd.to_numeric(out[column], errors="coerce").fillna(0).mean() * 100), 2),
    } for column in clinical_binary_cols + flag_cols])

    completeness = pd.DataFrame([{
        "Variable": column,
        "Nulos": int(out[column].isna().sum()),
        "Pct_Nulos": round(float(out[column].isna().mean() * 100), 2),
    } for column in out.columns])

    duplicate_matching = (
        trace.dropna(subset=["clave_matching"])
        .groupby("clave_matching", as_index=False)
        .agg(
            registros=("REGISTRO_ID", "count"),
            pacientes=("PACIENTE_ID", "nunique"),
        )
        .query("registros > 1 or pacientes > 1")
        .sort_values(["registros", "pacientes"], ascending=False)
    )

    detected_columns = pd.DataFrame([
        ["Columna nombre", detected["name_col"] or "NO_DETECTADA"],
        ["Columna RUT", detected["rut_col"] or "NO_DETECTADA"],
        ["Columna fecha nacimiento", detected["birth_col"] or "NO_DETECTADA"],
        ["Columna fecha atención", detected["attention_date_col"] or "NO_DETECTADA"],
    ], columns=["Elemento", "Valor"])

    sheets = {
        "Cohorte_Anonimizada": out,
        "Resumen_Validacion": summary,
        "Anonimizacion": anonymization,
        "Diccionario_Clinico": dictionary,
        "Conteos_Variables": counts,
        "Completitud": completeness,
    }

    with pd.ExcelWriter(OUTPUT_XLSX, engine="xlsxwriter") as writer:
        for sheet_name, data in sheets.items():
            data.to_excel(writer, sheet_name=sheet_name, index=False)

    dictionary.to_csv(DICT_CSV, index=False, encoding="utf-8-sig")

    with pd.ExcelWriter(CROSSWALK_XLSX, engine="xlsxwriter") as writer:
        trace[[
            "REGISTRO_ID", "PACIENTE_ID", "nombre_paciente_norm", "fecha_atencion_norm",
            "clave_matching", "clave_hash_privada", "estado_validacion", "metodo_match", "observaciones"
        ]].to_excel(writer, sheet_name="crosswalk", index=False)
        duplicate_matching.to_excel(writer, sheet_name="duplicados_clave", index=False)
        detected_columns.to_excel(writer, sheet_name="columnas_detectadas", index=False)

    with pd.ExcelWriter(AUDITORIA_PRIVADA_XLSX, engine="xlsxwriter") as writer:
        trace.to_excel(writer, sheet_name="trazabilidad_privada", index=False)
        duplicate_matching.to_excel(writer, sheet_name="duplicados_clave", index=False)
        detected_columns.to_excel(writer, sheet_name="columnas_detectadas", index=False)

    LOG_TXT.write_text(
        "\n".join([
            "LOG DE TRANSFORMACION - COHORTE TFM",
            f"Fecha generación: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
            f"Filas originales: {len(df)}",
            f"Columnas originales: {df.shape[1]}",
            f"Filas finales: {len(out)}",
            f"Columnas finales: {out.shape[1]}",
            f"Pacientes únicos: {out['PACIENTE_ID'].nunique()}",
            f"Registros únicos: {out['REGISTRO_ID'].nunique()}",
            f"Identificadores directos eliminados: {', '.join(direct_identifiers)}",
            "Texto original AntecedentesMedicos conservado: No",
            f"Filas preservadas: {len(df) == len(out)}",
            f"Identificadores directos presentes en salida: {[c for c in direct_identifiers + ['AntecedentesMedicos'] if c in out.columns]}",
            f"Crosswalk privado generado: {CROSSWALK_XLSX}",
            f"Registros con clave_matching: {int(trace['clave_matching'].notna().sum())}",
            f"Duplicados clave_matching detectados: {len(duplicate_matching)}",
            "Advertencia: crosswalk_paciente_ecg.xlsx y Auditoria_Privada_Trazabilidad_TFM.xlsx no deben publicarse.",
        ]),
        encoding="utf-8"
    )

    return sheets, duplicate_matching


## 8. Ejecución del proceso

Esta celda ejecuta el pipeline completo. Requiere que `Base de Datos Original.xlsx` exista en el mismo directorio de ejecución.


In [10]:
if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró {INPUT_PATH}. Copia la base clínica original en el directorio del notebook antes de ejecutar."
    )

df_original, df_procesada, crosswalk_privado, identificadores, columnas_detectadas = build_cohort(INPUT_PATH)
sheets, duplicados_clave = export_outputs(
    df_original,
    df_procesada,
    crosswalk_privado,
    identificadores,
    columnas_detectadas,
)

print("Proceso 1 revisado ejecutado.")
print("Filas originales:", len(df_original))
print("Filas finales:", len(df_procesada))
print("Pacientes únicos:", df_procesada["PACIENTE_ID"].nunique())
print("Registros con clave_matching:", int(crosswalk_privado["clave_matching"].notna().sum()))
print("Duplicados clave_matching:", len(duplicados_clave))
print("Identificadores directos eliminados:", identificadores)
print("Archivo analítico generado:", OUTPUT_XLSX.resolve())
print("Crosswalk privado generado:", CROSSWALK_XLSX.resolve())
print("Log generado:", LOG_TXT.resolve())


Proceso 1 revisado ejecutado.
Filas originales: 3779
Filas finales: 3779
Pacientes únicos: 3779
Registros con clave_matching: 3773
Duplicados clave_matching: 0
Identificadores directos eliminados: ['Nombre', 'Rut', 'FechaNacimiento']
Archivo analítico generado: E:\Capacitacion 2025\Magister en Inteligencia Artificial\99 Trabajo Fin de Estudios TFE\GitHub\tfm-ecg-riesgo-cardiometabolico\notebooks\Base_Datos_Original_Anonimizada_Procesada_TFM.xlsx
Crosswalk privado generado: E:\Capacitacion 2025\Magister en Inteligencia Artificial\99 Trabajo Fin de Estudios TFE\GitHub\tfm-ecg-riesgo-cardiometabolico\notebooks\crosswalk_paciente_ecg.xlsx
Log generado: E:\Capacitacion 2025\Magister en Inteligencia Artificial\99 Trabajo Fin de Estudios TFE\GitHub\tfm-ecg-riesgo-cardiometabolico\notebooks\Log_Transformacion_Cohorte_TFM.txt


## 9. Validaciones estructurales mínimas

Estas validaciones bloquean salidas inseguras o inconsistentes.


In [11]:
# Validación estructural mínima
assert len(df_original) == len(df_procesada), "La cantidad de filas no fue preservada."
assert "PACIENTE_ID" in df_procesada.columns, "No existe PACIENTE_ID."
assert "REGISTRO_ID" in df_procesada.columns, "No existe REGISTRO_ID."
assert df_procesada["REGISTRO_ID"].is_unique, "REGISTRO_ID debe ser único."
assert not any(c in df_procesada.columns for c in identificadores + ["AntecedentesMedicos"]), "Persisten identificadores o texto sensible en la salida analítica."
assert CROSSWALK_XLSX.exists(), "No se generó el crosswalk privado."

required_crosswalk_cols = {
    "REGISTRO_ID", "PACIENTE_ID", "nombre_paciente_norm", "fecha_atencion_norm",
    "clave_matching", "clave_hash_privada", "estado_validacion", "metodo_match", "observaciones"
}
assert required_crosswalk_cols.issubset(set(crosswalk_privado.columns)), "Faltan columnas requeridas en crosswalk privado."

print("Validación estructural: OK")
print(df_procesada.head())


Validación estructural: OK
  REGISTRO_ID PACIENTE_ID  UltimaAtencion_Anio  UltimaAtencion_Mes  DiasDesdePrimeraAtencion  Edad       Sexo   Peso    Altura    IMC PA_Sistolica PA_Diastolica  Glicemia ColesterolTotal       HDL  \
0  REG_000001  PAC_000001               2025.0                 7.0                    1003.0    22  Masculino  95 Kg  1.70 mts  32.87    138 mm/Hg      74 mm/Hg  98 mg/dl               -         -   
1  REG_000002  PAC_000002               2026.0                 2.0                    1227.0    48  Masculino  79 Kg  1.72 mts  26.70    135 mm/Hg      85 mm/Hg  89 mg/dl       244 mg/dl  47 mg/dl   
2  REG_000003  PAC_000003               2023.0                12.0                     423.0    43  Masculino      -         -      -            -             -         -               -         -   
3  REG_000004  PAC_000004               2024.0                 5.0                     580.0    28  Masculino  65 Kg  1.69 mts  22.76    118 mm/Hg      64 mm/Hg  80 mg/dl   

## 10. Validación específica para integración ECG

La integración posterior solo debe ejecutarse si el crosswalk contiene claves suficientes y revisadas. Los duplicados no implican error automático, pero deben auditarse antes del notebook 04.


In [12]:
registros_con_clave = int(crosswalk_privado["clave_matching"].notna().sum())
registros_totales = len(crosswalk_privado)
pct_con_clave = round(registros_con_clave / registros_totales * 100, 2) if registros_totales else 0

print("Cobertura clave_matching:", f"{pct_con_clave}%", f"({registros_con_clave}/{registros_totales})")

if registros_con_clave == 0:
    raise ValueError(
        "No se generó ninguna clave_matching. Revisar columnas de nombre y fecha de atención detectadas."
    )

if len(duplicados_clave) > 0:
    print("Advertencia: existen claves duplicadas. Revisar hoja 'duplicados_clave' en crosswalk_paciente_ecg.xlsx.")
else:
    print("No se detectaron duplicados en clave_matching.")


Cobertura clave_matching: 99.84% (3773/3779)
No se detectaron duplicados en clave_matching.


## 11. Consideraciones de privacidad

No se publica en GitHub:

```text
Base de Datos Original.xlsx
crosswalk_paciente_ecg.xlsx
Auditoria_Privada_Trazabilidad_TFM.xlsx
PDFs ECG originales
archivos derivados que contengan nombre, RUT, fecha de nacimiento o clave_matching basada en nombre
```

Sí se publica, solo si no contiene datos reales:

```text
notebooks
scripts
diccionarios
logs anonimizados
README
archivos sintéticos de ejemplo
```
